# Chapter 16 &mdash; Clique is NP-Complete: Reduction from 3-SAT

**Concept 9 of the Chapter 16 decomposition:** *Clique is NP-Complete: Reduction from 3-SAT*

One island of three nodes per clause; connect compatible literals across islands; $k$ = number of clauses.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter16/Concept-Clique-Is-NPC/Concept-Clique-Is-NPC.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
# Run me first.  Works on Colab and on a local Jove checkout.
import os, subprocess, sys

def _git(*a):
    r = subprocess.run(('git',) + a, capture_output=True, text=True)
    return r.stdout.strip() if r.returncode == 0 else ''

REPO = 'https://github.com/ganeshutah/Jove'
try:                       # ---- Colab: clone once, pull thereafter ----
    import google.colab
    was = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD') if os.path.isdir('Jove') else ''
    if os.path.isdir('Jove') and not was:
        print('Jove: WARNING ./Jove exists but is not a git checkout -- left as is')
    elif was:
        _git('-C', 'Jove', 'pull', '-q', '--ff-only')
        now = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD')
        if now and now != was:
            print('Jove: PULLED  %s -> %s' % (was, now))
            print(_git('-C', 'Jove', 'log', '--oneline', was + '..' + now))
        else:
            print('Jove: PULLED  already current at %s' % (now or was))
    else:
        _git('clone', '-q', REPO, 'Jove')
        print('Jove: CLONED  at %s' % (_git('-C', 'Jove', 'rev-parse',
                                             '--short', 'HEAD') or '?'))
    JOVE = 'Jove'
except ImportError:        # ---- local: the checkout above Chapter<N>/ ----
    JOVE = next((p for p in ('../..', '../../..', '..', '.')
                 if os.path.isdir(os.path.join(p, 'jove'))), '../..')
    print('Jove: LOCAL   checkout at %s'
          % (_git('-C', JOVE, 'rev-parse', '--short', 'HEAD') or '?'))
sys.path.insert(0, JOVE)

# A session can already hold an OLDER jove in sys.modules.  The pull above
# updates the files on disk, but `import` would hand back the cached module --
# so a fixed library still behaves like the broken one.  Drop them first.
for _m in [k for k in list(sys.modules) if k == 'jove' or k.startswith('jove.')]:
    del sys.modules[_m]

from jove.Def_md2mc      import *
from jove.DotBashers     import *
from jove.Def_DFA        import *
from jove.Def_NFA        import *
from jove.LangDef        import *

import jove; print('Jove loaded from', list(jove.__path__)[0])

## 1. The idea


The reduction 3-SAT $\le_p$ **Clique**, which is the prettiest one in the chapter.

Given a 3-CNF with $k$ clauses:

* make **one island of three nodes per clause**, each node labelled by a literal;
* join two nodes **iff** they are in **different islands** and their literals are
  **compatible** (not $x$ and $\neg x$);
* ask for a clique of size $k$.

A $k$-clique must take exactly **one node per island** (nodes inside an island are
never joined) and its literals are mutually compatible &mdash; so setting them all true
satisfies every clause. Conversely a satisfying assignment picks one true literal per
clause, and those nodes form a $k$-clique.

The construction is clearly polynomial: $3k$ nodes and at most $\binom{3k}{2}$ edges.

## 2. Definitions

### The reduction

In [ ]:
# --- a tiny CNF toolkit -------------------------------------------------
# A literal is an int: 3 means x3, -3 means NOT x3.
# A clause is a tuple of literals; a formula is a list of clauses.
from itertools import product

def nvars(F):
    return max((abs(l) for c in F for l in c), default=0)

def evaluate(F, assign):
    # assign: dict var -> bool
    return all(any(assign[abs(l)] == (l > 0) for l in c) for c in F)

def brute_sat(F):
    n = nvars(F)
    for bits in product([False, True], repeat=n):
        a = {i + 1: bits[i] for i in range(n)}
        if evaluate(F, a): return a
    return None

def show_cnf(F):
    def lit(l): return ("x%d" % l) if l > 0 else ("~x%d" % -l)
    return " AND ".join("(" + " OR ".join(lit(l) for l in c) + ")" for c in F)


def sat_to_clique(F):
    nodes = [(i, l) for i, c_ in enumerate(F) for l in c_]
    edges = set()
    for a in range(len(nodes)):
        for b in range(a + 1, len(nodes)):
            (ia, la), (ib, lb) = nodes[a], nodes[b]
            if ia != ib and la != -lb:
                edges.add((a, b))
    return nodes, edges, len(F)

### A clique finder, and the translation back to an assignment

In [ ]:
import itertools
def find_clique(nodes, edges, k):
    E = set(edges) | {(b, a) for a, b in edges}
    for comb in itertools.combinations(range(len(nodes)), k):
        if all((a, b) in E for a, b in itertools.combinations(comb, 2)):
            return comb
    return None

def clique_to_assignment(nodes, clique, n):
    a = {v: False for v in range(1, n + 1)}
    for i in clique:
        l = nodes[i][1]
        a[abs(l)] = (l > 0)
    return a

<!-- nav-strip -->

---

&larr;&nbsp;[Ch16&nbsp;8.&nbsp;The Cook–Levin Theorem: 3-SAT is NP-Complete](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter16/Concept-Cook-Levin/Concept-Cook-Levin.ipynb) &nbsp;&middot;&nbsp; [**Chapter 16** index](https://github.com/ganeshutah/Jove/blob/master/Chapter16/README.md) &nbsp;&middot;&nbsp; [Ch16&nbsp;10.&nbsp;NP-Hard Can Be Undecidable: the Diophantine Pitfall](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter16/Concept-NP-Hard-Can-Be-Undecidable/Concept-NP-Hard-Can-Be-Undecidable.ipynb)&nbsp;&rarr;

---

## 3. Tests

A satisfiable 3-CNF, and the graph it becomes.

In [ ]:
F = [(1, -2, 3), (-1, 2, 3), (1, 2, -3)]
print(show_cnf(F))
nodes, edges, k = sat_to_clique(F)
print("\nnodes (%d) :" % len(nodes), nodes)
print("k = %d (one per clause), edges = %d" % (k, len(edges)))
assert len(nodes) == 3 * len(F)

**No edges inside an island**, so a $k$-clique takes one node per clause.

In [ ]:
inside = [(a, b) for a, b in edges if nodes[a][0] == nodes[b][0]]
print("intra-island edges :", inside)
assert not inside

The clique exists, and it decodes to a satisfying assignment.

In [ ]:
cl = find_clique(nodes, edges, k)
print("clique :", cl, "->", [nodes[i] for i in cl])
assert cl is not None
a = clique_to_assignment(nodes, cl, nvars(F))
print("assignment :", a)
print("satisfies F? ", evaluate(F, a))
assert evaluate(F, a)

An **unsatisfiable** formula gives a graph with no $k$-clique.

In [ ]:
U = [tuple(v * s for v, s in zip((1, 2, 3), signs))
     for signs in product([1, -1], repeat=3)]
print("unsatisfiable formula with %d clauses" % len(U))
nU, eU, kU = sat_to_clique(U)
print("looking for a %d-clique among %d nodes ..." % (kU, len(nU)))
print("found :", find_clique(nU, eU, kU))
assert brute_sat(U) is None
assert find_clique(nU, eU, kU) is None

The correspondence holds on random instances, both directions.

In [ ]:
import random
def random_3sat(nv, nc, seed):
    random.seed(seed)
    return [tuple(v * random.choice([1, -1])
                  for v in random.sample(range(1, nv + 1), 3))
            for _ in range(nc)]
bad = 0
for s in range(25):
    F = random_3sat(4, 4, s)
    n_, e_, k_ = sat_to_clique(F)
    cl = find_clique(n_, e_, k_)
    if (cl is not None) != (brute_sat(F) is not None): bad += 1
    if cl is not None:
        assert evaluate(F, clique_to_assignment(n_, cl, nvars(F)))
print("25 random 3-CNFs : %d mismatches between SAT and k-clique" % bad)
assert bad == 0

And the construction is **polynomial**.

In [ ]:
print("%-8s %-8s %-10s" % ("clauses", "nodes", "edges (max)"))
for nc in [3, 10, 40, 100]:
    print("%-8d %-8d %-10d" % (nc, 3 * nc, (3 * nc) * (3 * nc - 1) // 2))
print("\nO(k) nodes, O(k^2) edges -- built without solving anything.")

## 4. Exercises


1. Draw the graph for $(x_1\vee x_2\vee x_3)\wedge(\neg x_1\vee \neg x_2 \vee x_3)$.
2. Why must the two nodes be in **different** islands to be joined?
3. Reduce Clique to Vertex Cover. What is the trick?

In [ ]:
# Your work for the exercises above.

## 5. Where next

In [ ]:
# Previous / next, and a search box for all 245 concepts.
# Type a chapter (Chapter7, ch7) or words from a title (pumping, subset).
#
# Following a link opens a NEW Colab runtime. To pull another concept's
# definitions into THIS session instead:  load_here('Chapter7/Concept-...')
from jove.Nav import nav, load_here
nav(here='Chapter16/Concept-Clique-Is-NPC')